# Klasifikasi DemogPairs Menggunakan ViT (Wajah) & Gaussian Naive Bayes

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
features = joblib.load('features/demogpairs_vit-face.pkl')
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

Jumlah fitur per gambar: 768


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
var_smoothing_values = np.logspace(-9, 2, 40)  # dari 1e-9 sampai 1e2, 40 nilai

grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [GaussianNB()],
        'classifier__var_smoothing': var_smoothing_values
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro'
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

GaussianNB: 240 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models,
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix='models/clf_demogpairs_gnb_vit-face_',
    results_path='results/demogpairs_gnb_vit-face_'
)

sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: GaussianNB


{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.04124626382901348), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}


Accuracy  : 0.8268518518518518
Precision : 0.8271037207967583
Recall    : 0.8268518518518517
F1 Score  : 0.8257926480465394
               precision    recall  f1-score   support

Asian_Females     0.8280    0.8556    0.8415       360
  Asian_Males     0.8819    0.8917    0.8867       360
Black_Females     0.7745    0.7917    0.7830       360
  Black_Males     0.8621    0.9028    0.8820       360
White_Females     0.8311    0.6972    0.7583       360
  White_Males     0.7851    0.8222    0.8033       360

     accuracy                         0.8269      2160
    macro avg     0.8271    0.8269    0.8258      2160
 weighted avg     0.8271    0.8269    0.8258      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9462962962962963,0.8279569892473119,0.8555555555555555,0.8415300546448087,360
Asian_Males,0.962037037037037,0.8818681318681318,0.8916666666666667,0.8867403314917127,360
Black_Females,0.9268518518518518,0.7744565217391305,0.7916666666666666,0.782967032967033,360
Black_Males,0.9597222222222223,0.8620689655172413,0.9027777777777778,0.8819538670284938,360
White_Females,0.9259259259259259,0.8311258278145696,0.6972222222222222,0.7583081570996979,360
White_Males,0.9328703703703703,0.7851458885941645,0.8222222222222222,0.8032564450474898,360


Confusion matrix saved: images\cm_gnb_vit-face_GaussianNB.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               308                 0                17                24                11                 0
         Asian_Males                 2               321                 5                 0                13                19
       Black_Females                 9                 0               285                22                 3                41
         Black_Males                 5                10                16               325                 2                 2
       White_Females                48                23                17                 2               251                19
         White_Males                 0                10                28                 4                22               296


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
GaussianNB,models/clf_demogpairs_gnb_vit-face_GaussianNB.pkl,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.04124626382901348), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8268518518518518,0.8257926480465394,0.8271037207967583,0.8268518518518517,240


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_gnb_vit-face_GaussianNB.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 474.0,
 'days': 0,
 'hours': 0,
 'minutes': 7,
 'seconds': 54.0,
 'text': '0 hari 0 jam 7 menit 54.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 2402.0,
 'days': 0,
 'hours': 0,
 'minutes': 40,
 'seconds': 2.0,
 'text': '0 hari 0 jam 40 menit 2.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.04124626382901348), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8495,0.831,0.8455,0.8403,0.8356,0.8404,0.8398,0.8414,0.8404,1.9178
2,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.07896522868499734), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8536,0.8328,0.8403,0.8362,0.8368,0.8399,0.8395,0.8416,0.8399,2.1884
3,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.151177507061566), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8553,0.8304,0.8362,0.8351,0.8403,0.8395,0.8391,0.8415,0.8395,2.2082
4,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.021544346900318822), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.853,0.8299,0.8426,0.8368,0.8304,0.8385,0.8376,0.8388,0.8385,1.8626
...,...,...,...,...,...,...,...,...,...,...,...
237,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(14.251026703029963), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8148,0.8079,0.805,0.8015,0.8015,0.8061,0.8021,0.8146,0.8061,2.7503
238,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(27.283333764867695), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8148,0.8079,0.8044,0.8021,0.8015,0.8061,0.802,0.8147,0.8061,2.8146
239,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(52.233450742668325), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8142,0.8084,0.8038,0.8021,0.8009,0.8059,0.8018,0.8146,0.8059,3.3981
240,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(100.0), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8131,0.8084,0.8038,0.8015,0.8009,0.8056,0.8014,0.8143,0.8056,2.3935
